In [1]:
from typing import Dict, TypedDict
from langgraph.graph import StateGraph

In [2]:
class AgentState(TypedDict):
    message: str

def greeting(state: AgentState) -> AgentState:
    """"Function to greet the user."""
    state["message"] = "Hello, how can I assist you today?"
    return {state}





In [5]:
import dotenv
import os
from src.utils.excel_processor import get_story_by_id
from src.prompts.prompt_manager import PromptManager
from src.utils.file_handler import save_text_file
from src.reviewers.reviewer import OopsPitfallReviewer
from google import genai
import langgraph
from langgraph.graph import StateGraph, END
from typing import Dict, Any, List, TypedDict

dotenv.load_dotenv()
llm = genai.Client()
def call_gemini(prompt: str) -> str:
    response = llm.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text
prompt_manager = PromptManager(prompts_file_path="src/prompts/prompts.yaml")

class OntoAgentState(TypedDict, total=False):
    story_id: str
    story_object: Any
    unprocessed_cqs: List[Any]
    processed_owls: List[Any]
    generated_owl: str
    combined_owl: str
    error_message: str
    is_valid: bool
    final_result: Any
    tool_calls: List[Any]
    current_cq: Any
    current_owl: str
    current_validation: Any
    validation_ok: bool
    combined_validation: Any
    combined_validation_ok: bool

# --- Node functions ---
def get_story_node(state: OntoAgentState) -> OntoAgentState:
    story_id = state["story_id"]
    story = get_story_by_id(story_id)
    return {
        **state,
        "story_object": story,
        "unprocessed_cqs": list(story.competency_questions),
        "processed_owls": [],
    }

def generate_owl_node(state: OntoAgentState) -> OntoAgentState:
    story = state["story_object"]
    cqs = state["unprocessed_cqs"]
    processed_owls = state["processed_owls"]
    cq = cqs[0]
    prompt = prompt_manager.get_structured_prompt("otho_memless_cq_by_cq")["task"].format(story=story.context, CQ=cq.question)
    llm_response = call_gemini(prompt)
    owl_code = llm_response.text if hasattr(llm_response, 'text') else llm_response
    return {
        **state,
        "current_cq": cq,
        "current_owl": owl_code
    }

def validate_owl_node(state: OntoAgentState) -> OntoAgentState:
    reviewer = OopsPitfallReviewer()
    try:
        validation_result = reviewer.review_owl_content(state["current_owl"])
        print("Validating CQ ID:", cq.id)
        
        return {
            **state,
            "current_validation": validation_result,
            "validation_ok": True
        }
    except Exception as e:
        return {
            **state,
            "current_validation": str(e),
            "validation_ok": False
        }

def store_owl_node(state: OntoAgentState) -> OntoAgentState:
    story_id = state["story_id"]
    cq = state["current_cq"]
    owl_code = state["current_owl"]
    processed_owls = state["processed_owls"] + [(cq.id, owl_code)]
    print("Storing CQ ID:", cq.id)
    save_text_file(f"data/output/{story_id}_{cq.id}.owl", owl_code)
    unprocessed_cqs = state["unprocessed_cqs"][1:]
    return {
        **state,
        "processed_owls": processed_owls,
        "unprocessed_cqs": unprocessed_cqs
    }

def combine_owls_node(state: OntoAgentState) -> OntoAgentState:
    story_id = state["story_id"]
    combined_owl = "\n".join([owl for _, owl in state["processed_owls"]])
    save_text_file(f"data/output/{story_id}_combined.owl", combined_owl)
    return {
        **state,
        "combined_owl": combined_owl
    }

def validate_combined_owl_node(state: OntoAgentState) -> OntoAgentState:
    reviewer = OopsPitfallReviewer()
    try:
        validation_result = reviewer.review_owl_content(state["combined_owl"])
        save_text_file(f"data/output/{state['story_id']}_combined_oops_result.xml", validation_result)
        return {
            **state,
            "combined_validation": validation_result,
            "combined_validation_ok": True
        }
    except Exception as e:
        return {
            **state,
            "combined_validation": str(e),
            "combined_validation_ok": False
        }

def end_node(state: OntoAgentState) -> OntoAgentState:
    print(f"Workflow complete for story {state['story_id']}")
    return state

# --- Graph definition ---
graph = StateGraph(state_schema=OntoAgentState)
graph.add_node('get_story', get_story_node)
graph.add_node('generate_owl', generate_owl_node)
graph.add_node('validate_owl', validate_owl_node)
graph.add_node('store_owl', store_owl_node)
graph.add_node('combine_owls', combine_owls_node)
graph.add_node('validate_combined_owl', validate_combined_owl_node)
graph.add_node('end', end_node)

graph.set_entry_point('get_story')

graph.add_edge('get_story', 'generate_owl')
graph.add_edge('generate_owl', 'validate_owl')
graph.add_conditional_edges('validate_owl', lambda s: 'store_owl' if s["validation_ok"] else 'generate_owl')
graph.add_conditional_edges('store_owl', lambda s: 'generate_owl' if len(s["unprocessed_cqs"]) > 0 else 'combine_owls')
graph.add_edge('combine_owls', 'validate_combined_owl')
graph.add_conditional_edges('validate_combined_owl', lambda s: 'end' if s["combined_validation_ok"] else 'combine_owls')

graph.set_finish_point('end')
# --- Run the workflow ---
app = graph.compile()
  

In [3]:
graph = StateGraph(AgentState)

graph.add_node(
    "greeting",	greeting)

graph.set_entry_point("greeting")
graph.set_finish_point("greeting")


app = graph.compile()

In [6]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 400.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`